# Clean Boundary Token Extraction
Direct model inference on raw corpus

In [ ]:
from google.colab import drive
import os, time
drive.mount('/content/drive')
time.sleep(2)
os.chdir('/content/drive/MyDrive/khabar-segmentation')
!pip install transformers torch --quiet
print('Setup OK')

In [ ]:
# Load corpus
with open('data/processed/kitab_uqala_reference_corpus.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(f'Corpus: {len(text):,} chars')

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

model_name = "CAMeL-Lab/bert-base-arabic-camelbert-msa"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print('Tokenizer loaded')

# Try to load from local checkpoint, else use base model
from pathlib import Path
local_model = 'checkpoints/camelbert_model'
if Path(local_model).exists():
    model = AutoModelForTokenClassification.from_pretrained(local_model)
    print(f'Model loaded from {local_model}')
else:
    # Load base model from HuggingFace
    model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=2)
    print('Model loaded from HuggingFace')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device).eval()
print(f'Device: {device}')

In [ ]:
# Tokenize corpus
tokens = tokenizer.tokenize(text)
print(f'Tokens: {len(tokens):,}')

# Encode for model
encoded = tokenizer.encode_plus(
    text,
    return_tensors='pt',
    truncation=False,
    padding=False,
    return_offsets_mapping=True
)
print(f'Input shape: {encoded["input_ids"].shape}')

In [ ]:
# Run inference
print('Running inference...')
with torch.no_grad():
    outputs = model(**{k: v.to(device) for k, v in encoded.items() if k != 'offset_mapping'})
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

predictions_np = predictions.cpu().numpy().flatten()
print(f'Predictions: {len(predictions_np):,}')
print(f'Boundary (1): {(predictions_np == 1).sum():,}')
print(f'Non-boundary (0): {(predictions_np == 0).sum():,}')

In [ ]:
# Map predictions to tokens
token_ids = encoded['input_ids'][0].cpu().numpy()
offset_mapping = encoded['offset_mapping'][0].cpu().numpy()

boundary_tokens = []
boundary_indices = []
token_details = []

for idx, (token_id, (char_start, char_end), pred) in enumerate(zip(token_ids, offset_mapping, predictions_np)):
    token_text = tokenizer.decode([token_id])
    token_details.append({
        'index': idx,
        'token': token_text,
        'prediction': int(pred),
        'char_start': int(char_start),
        'char_end': int(char_end),
    })
    if pred == 1:
        boundary_tokens.append(token_text)
        boundary_indices.append(idx)

print(f'Boundary tokens extracted: {len(boundary_tokens):,}')
print(f'Percentage: {100 * len(boundary_tokens) / len(token_details):.2f}%')

In [ ]:
# Preview
print('First 30 boundary tokens:')
for i, token in enumerate(boundary_tokens[:30], 1):
    print(f'{i:2d}. {token}')

In [ ]:
import json
from datetime import datetime

# Create results
results = {
    'metadata': {
        'corpus': 'kitab_uqala_reference_corpus.txt',
        'corpus_size_chars': len(text),
        'corpus_size_tokens': len(token_details),
        'model': model_name,
        'extraction_method': 'Direct model inference',
        'timestamp': datetime.now().isoformat(),
    },
    'statistics': {
        'total_tokens': len(token_details),
        'boundary_tokens_count': len(boundary_tokens),
        'non_boundary_tokens': len(token_details) - len(boundary_tokens),
        'boundary_percentage': round(100 * len(boundary_tokens) / len(token_details), 2),
    },
    'boundary_tokens': boundary_tokens,
    'boundary_indices': boundary_indices,
}

# Save
output_path = 'results/camelbert_boundary_tokens_clean.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

import os
file_size = os.path.getsize(output_path) / 1024
print(f'Saved: {output_path}')
print(f'Size: {file_size:.1f} KB')
print(f'Boundary tokens: {len(boundary_tokens):,}')